# SmolVLA Efficiency Tricks

**Lecture 3, Part 3 — Vizuara Modern Robot Learning Bootcamp**

SmolVLA turns a 7B-parameter VLA into a **500M-parameter** model that runs on edge devices. How? Three core tricks:

| Trick | What It Does | Savings |
|-------|-------------|--------|
| **PixelShuffle** | Compress 256 vision tokens to 64 | 16x attention reduction |
| **Layer Skipping** | Use bottom N/2 VLM layers only | 2x forward pass |
| **Async Inference** | Overlap encoding with action execution | ~30% wall-clock |

We build each trick from scratch, benchmark it, then wire them all together into a complete SmolVLA forward pass.

**Running example:** An image processing pipeline with CIFAR-100 images upscaled to 512x512, simulating the real-world cost of high-resolution robot camera inputs.

---

In [ ]:
!pip install -q torch torchvision numpy matplotlib 2>&1 | tail -3

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time
import math

# Vizuara color palette
ACCENT = '#D97757'
BLUE   = '#5B8CB8'
TEAL   = '#7DA488'
PURPLE = '#9B7EC8'
WARM   = '#C4956A'
BG     = '#FAF8F5'

plt.rcParams['figure.facecolor'] = BG
plt.rcParams['axes.facecolor']   = BG
plt.rcParams['font.size']        = 12

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

---
## Step 1: Setup + The Efficiency Problem

Vision transformers turn images into **sequences of tokens**. SigLIP with 14x14 patches on a 224x224 image produces 256 tokens. Self-attention over those tokens costs O(n^2) — every token attends to every other token.

Let's load a real image and compute just how expensive this gets.

In [ ]:
# Load a CIFAR-100 image and upscale to 512x512 (simulating high-res robot camera)
try:
    from torchvision.datasets import CIFAR100
    from torchvision import transforms
    dataset = CIFAR100(root='/tmp/cifar100', train=True, download=True)
    raw_img = dataset[0][0]  # PIL Image 32x32
    img_512 = raw_img.resize((512, 512))
    img_tensor = transforms.ToTensor()(img_512)  # [3, 512, 512]
    print(f"Loaded CIFAR-100 image, upscaled to {img_512.size}")
except Exception:
    # Fallback: synthetic image
    img_tensor = torch.rand(3, 512, 512)
    print("Created synthetic 512x512 image")

# Show the image
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(img_tensor.permute(1, 2, 0).numpy().clip(0, 1))
ax.set_title('Input Image (512x512)', fontsize=14, color=ACCENT, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

# Compute O(n^2) attention cost for different sequence lengths
seq_lengths = [64, 128, 256, 512, 1024]
attention_ops = [n * n for n in seq_lengths]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar([str(n) for n in seq_lengths], attention_ops, color=ACCENT, edgecolor='white', linewidth=1.5)
ax.set_xlabel('Sequence Length (tokens)', fontweight='bold')
ax.set_ylabel('Attention Operations (n^2)', fontweight='bold')
ax.set_title('Self-Attention Cost Grows Quadratically', fontsize=14, color=ACCENT, fontweight='bold')

# Annotate bars
for bar, ops in zip(bars, attention_ops):
    label = f'{ops:,}'
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10000,
            label, ha='center', va='bottom', fontsize=10, fontweight='bold', color=BLUE)

# Add quadratic curve
xs = np.linspace(0, 4, 100)
ys = np.interp(xs, range(len(seq_lengths)), attention_ops)
ax.plot(xs, ys, '--', color=BLUE, linewidth=2, alpha=0.5)

ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

print(f"256 tokens: {256*256:,} operations. 1024 tokens: {1024*1024:,} operations. 16x more tokens = 256x more compute.")
print(f"\nSmolVLA's goal: cut 256 tokens down to 64 tokens = 16x less attention cost.")

---
## Step 2: PixelShuffle — Spatial Token Compression

PixelShuffle (originally from super-resolution) rearranges spatial pixels into channel dimensions. SmolVLA uses the **reverse** operation: it takes a 16x16 grid of vision tokens and packs 2x2 neighborhoods into single tokens with 4x the channel depth.

Result: 256 tokens (16x16) become 64 tokens (8x8), each carrying 4x more information.

In [ ]:
class PixelShuffleCompressor(nn.Module):
    """Compress spatial tokens via PixelShuffle (reverse / unshuffle).
    
    Takes patch tokens [B, H*W, D] with H=W=16 (256 patches)
    and merges 2x2 neighborhoods into single tokens with 4x channels.
    Output: [B, (H/2)*(W/2), D*4] = [B, 64, D*4]
    """
    def __init__(self, d_model, grid_size=16, downsample_factor=2):
        super().__init__()
        self.grid_size = grid_size
        self.r = downsample_factor
        self.d_model = d_model
        # Output dimension = d_model * r^2
        self.out_dim = d_model * (self.r ** 2)
    
    def forward(self, x):
        # x: [B, H*W, D] where H*W = grid_size^2
        B, N, D = x.shape
        H = W = self.grid_size
        r = self.r
        
        # Reshape to spatial grid: [B, H, W, D]
        x = x.view(B, H, W, D)
        
        # Split into r x r blocks: [B, H//r, r, W//r, r, D]
        x = x.view(B, H // r, r, W // r, r, D)
        
        # Merge the r x r spatial elements into the channel dim
        # Permute: [B, H//r, W//r, r, r, D] then flatten last 3 dims
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
        x = x.view(B, (H // r) * (W // r), r * r * D)
        
        return x  # [B, 64, D*4]


# Demo: compress 256 tokens to 64
D = 384  # typical SigLIP patch dimension (for illustration)
compressor = PixelShuffleCompressor(d_model=D, grid_size=16, downsample_factor=2)

# Create dummy patch tokens
tokens_in = torch.randn(1, 256, D)
tokens_out = compressor(tokens_in)

print(f"Input:  {list(tokens_in.shape)}  — 16x16 = 256 patches, each {D}-d")
print(f"Output: {list(tokens_out.shape)} — 8x8 = 64 patches, each {D*4}-d")
print(f"\nTotal information preserved: {256 * D} = {64 * D * 4} values (lossless reshape!)")

# Visualize: 16x16 grid -> 8x8 grid transformation
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Before: 16x16 grid with colored 2x2 blocks
ax = axes[0]
grid_before = np.zeros((16, 16, 3))
colors_rgb = {
    'accent': np.array([0.85, 0.47, 0.34]),
    'blue':   np.array([0.36, 0.55, 0.72]),
    'teal':   np.array([0.49, 0.64, 0.53]),
    'purple': np.array([0.61, 0.49, 0.78]),
}
color_list = list(colors_rgb.values())
for i in range(8):
    for j in range(8):
        c = color_list[(i * 8 + j) % len(color_list)]
        grid_before[i*2,   j*2]   = c
        grid_before[i*2,   j*2+1] = c
        grid_before[i*2+1, j*2]   = c
        grid_before[i*2+1, j*2+1] = c

ax.imshow(grid_before, interpolation='nearest')
ax.set_title('Before: 16x16 Grid (256 tokens)', fontsize=13, color=ACCENT, fontweight='bold')
ax.set_xlabel('Each 2x2 block shares a color', fontweight='bold')
# Draw grid lines
for i in range(17):
    ax.axhline(i - 0.5, color='white', linewidth=0.5)
    ax.axvline(i - 0.5, color='white', linewidth=0.5)
# Bold lines for 2x2 blocks
for i in range(0, 17, 2):
    ax.axhline(i - 0.5, color='white', linewidth=2)
    ax.axvline(i - 0.5, color='white', linewidth=2)
ax.set_xticks([]); ax.set_yticks([])

# After: 8x8 grid — each cell now contains merged info
ax = axes[1]
grid_after = np.zeros((8, 8, 3))
for i in range(8):
    for j in range(8):
        c = color_list[(i * 8 + j) % len(color_list)]
        grid_after[i, j] = c

ax.imshow(grid_after, interpolation='nearest')
ax.set_title('After: 8x8 Grid (64 tokens, 4x channels)', fontsize=13, color=TEAL, fontweight='bold')
ax.set_xlabel('2x2 neighbors merged into channel dim', fontweight='bold')
for i in range(9):
    ax.axhline(i - 0.5, color='white', linewidth=2)
    ax.axvline(i - 0.5, color='white', linewidth=2)
ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('PixelShuffle Compression: Spatial -> Channel', fontsize=15, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

# Verify spatial info is preserved: neighboring patches are packed together
print("\nSpatial info preservation check:")
print(f"  Token (0,0) in output contains info from patches (0,0), (0,1), (1,0), (1,1)")
print(f"  Token (0,1) in output contains info from patches (0,2), (0,3), (1,2), (1,3)")
print(f"  -> Neighboring spatial info stays together, just packed into channels")

# Benchmark attention FLOPs
flops_256 = 256 * 256 * D  # Q @ K^T for 256 tokens
flops_64  = 64 * 64 * (D * 4)  # Q @ K^T for 64 tokens (higher dim cancels in FLOPs comparison)
# Actually for attention: FLOPs = 2 * n^2 * d (for Q@K^T) + 2 * n^2 * d (for attn@V)
# The key comparison is n^2
print(f"\nAttention cost comparison (n^2 term):")
print(f"  256 tokens: {256**2:>8,} operations")
print(f"   64 tokens: {64**2:>8,} operations")
print(f"  Reduction:  {256**2 / 64**2:.0f}x fewer attention operations")

---
## Step 3: Layer Skipping

In a deep transformer, the bottom layers learn fast-changing visual features (edges, textures). The top layers refine semantic meaning slowly. SmolVLA's insight: for robotics, we only need the bottom half of the VLM backbone.

Let's build a 12-layer toy transformer and measure how much the representations change in each layer.

In [ ]:
class TransformerBlock(nn.Module):
    """Pre-LN Transformer block."""
    def __init__(self, d_model=256, n_heads=4, d_ff=1024):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
    
    def forward(self, x):
        # Pre-LN: normalize before attention/FF
        h = self.ln1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        h = self.ln2(x)
        x = x + self.ff(h)
        return x


class ToyVLMBackbone(nn.Module):
    """12-layer transformer simulating a VLM backbone."""
    def __init__(self, n_layers=12, d_model=256, n_heads=4, d_ff=1024):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff)
            for _ in range(n_layers)
        ])
    
    def forward(self, x, return_intermediates=False):
        intermediates = []
        for layer in self.layers:
            x = layer(x)
            if return_intermediates:
                intermediates.append(x.detach())
        if return_intermediates:
            return x, intermediates
        return x


# Build the 12-layer backbone
d_model = 256
n_layers = 12
backbone = ToyVLMBackbone(n_layers=n_layers, d_model=d_model).to(device)
backbone.eval()

n_params = sum(p.numel() for p in backbone.parameters())
print(f"Toy VLM Backbone: {n_layers} layers, d_model={d_model}, params={n_params:,}")

# Pass a batch of token sequences through all 12 layers
B, seq_len = 4, 64  # 4 sequences of 64 tokens each
x_input = torch.randn(B, seq_len, d_model, device=device)

with torch.no_grad():
    final_output, intermediates = backbone(x_input, return_intermediates=True)

print(f"Input:  {list(x_input.shape)}")
print(f"Output: {list(final_output.shape)}")
print(f"Collected {len(intermediates)} intermediate representations")

# Measure cosine similarity between each layer's output and the final layer output
final_flat = final_output.view(B, -1)  # [B, seq_len * d_model]
similarities = []
for i, inter in enumerate(intermediates):
    inter_flat = inter.view(B, -1)
    cos_sim = F.cosine_similarity(inter_flat, final_flat, dim=-1).mean().item()
    similarities.append(cos_sim)
    print(f"  Layer {i+1:>2d} vs Final: cosine similarity = {cos_sim:.4f}")

# Plot: layer vs cosine similarity with final output
fig, ax = plt.subplots(figsize=(10, 5))
layers_x = list(range(1, n_layers + 1))

# Color: bottom half (keep) in teal, top half (skip) in lighter color
colors = [TEAL if i < n_layers // 2 else WARM for i in range(n_layers)]
ax.bar(layers_x, similarities, color=colors, edgecolor='white', linewidth=1.5)

ax.set_xlabel('Layer', fontweight='bold')
ax.set_ylabel('Cosine Similarity with Final Layer', fontweight='bold')
ax.set_title('Representation Change Per Layer', fontsize=14, color=ACCENT, fontweight='bold')
ax.set_xticks(layers_x)

# Add skip boundary
ax.axvline(6.5, color=ACCENT, linewidth=2, linestyle='--', label='Skip boundary')
ax.legend(fontsize=11)

# Legend patches
keep_patch = mpatches.Patch(color=TEAL, label='Keep (bottom 6)')
skip_patch = mpatches.Patch(color=WARM, label='Skip (top 6)')
ax.legend(handles=[keep_patch, skip_patch, ax.lines[0]], fontsize=10, loc='lower right')

ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

print(f"\nBottom layers (1-6): Features change rapidly — building visual representations.")
print(f"Top layers (7-12): Features refine slowly — high-level semantic reasoning.")
print(f"SmolVLA insight: For robot actions, bottom-half features are enough!")

# Compare: full 12 layers vs bottom 6 only
with torch.no_grad():
    x_full = x_input.clone()
    for layer in backbone.layers:
        x_full = layer(x_full)
    
    x_half = x_input.clone()
    for layer in backbone.layers[:n_layers // 2]:
        x_half = layer(x_half)

cos_sim_half_vs_full = F.cosine_similarity(
    x_half.view(B, -1), x_full.view(B, -1), dim=-1
).mean().item()

print(f"\nFull 12 layers vs bottom 6 layers:")
print(f"  Cosine similarity: {cos_sim_half_vs_full:.4f}")
print(f"  Compute saved: {n_layers // 2}/{n_layers} = {(n_layers // 2) / n_layers * 100:.0f}% of layers skipped")

---
## Step 4: Combined Efficiency Gains

Let's chain PixelShuffle + layer skipping and compute the total savings in a single visualization.

In [ ]:
# Compute FLOPs for each configuration
# Attention FLOPs per layer ~= 2 * n^2 * d (QK^T) + 2 * n^2 * d (attn@V) + 2 * n * d * d_ff (FFN)
# Simplified: dominant cost is n^2 * d for attention + n * d * d_ff for FFN

def estimate_flops(n_tokens, d_model, d_ff, n_layers):
    """Rough FLOP estimate for transformer forward pass."""
    attn_flops = 4 * n_tokens * n_tokens * d_model  # Q@K^T + attn@V (2 matmuls, each 2*n^2*d)
    ffn_flops  = 4 * n_tokens * d_model * d_ff       # 2 linear layers (each 2*n*d*d_ff)
    per_layer  = attn_flops + ffn_flops
    return per_layer * n_layers

# Baseline: 256 tokens, 12 layers
flops_baseline = estimate_flops(256, d_model, 1024, 12)

# After PixelShuffle: 64 tokens, 12 layers (d increases but attention dominates)
flops_pixelshuffle = estimate_flops(64, d_model, 1024, 12)

# After Layer Skip: 256 tokens, 6 layers
flops_layerskip = estimate_flops(256, d_model, 1024, 6)

# Combined: 64 tokens, 6 layers
flops_combined = estimate_flops(64, d_model, 1024, 6)

# Stacked bar chart
fig, ax = plt.subplots(figsize=(10, 6))

configs = ['Baseline\n(256 tok, 12 layers)',
           'PixelShuffle\n(64 tok, 12 layers)',
           'Layer Skip\n(256 tok, 6 layers)',
           'Combined\n(64 tok, 6 layers)']
flops_values = [flops_baseline, flops_pixelshuffle, flops_layerskip, flops_combined]
bar_colors = [WARM, BLUE, TEAL, ACCENT]

bars = ax.bar(configs, flops_values, color=bar_colors, edgecolor='white', linewidth=2)

# Annotate with speedup factor
for i, (bar, flops) in enumerate(zip(bars, flops_values)):
    speedup = flops_baseline / flops
    label = f'{flops / 1e6:.1f}M FLOPs\n({speedup:.0f}x)' if speedup > 1 else f'{flops / 1e6:.1f}M FLOPs\n(baseline)'
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + flops_baseline * 0.02,
            label, ha='center', va='bottom', fontsize=11, fontweight='bold', color=bar_colors[i])

ax.set_ylabel('FLOPs (forward pass)', fontweight='bold')
ax.set_title('Combined Efficiency Gains', fontsize=15, color=ACCENT, fontweight='bold')
ax.grid(True, alpha=0.2, axis='y')
ax.set_ylim(0, flops_baseline * 1.25)
plt.tight_layout()
plt.show()

total_speedup = flops_baseline / flops_combined
print(f"Baseline FLOPs:  {flops_baseline:>12,}")
print(f"Combined FLOPs:  {flops_combined:>12,}")
print(f"")
print(f"PixelShuffle:    {flops_baseline / flops_pixelshuffle:.0f}x attention reduction (256 -> 64 tokens)")
print(f"Layer Skip:      {flops_baseline / flops_layerskip:.0f}x forward pass reduction (12 -> 6 layers)")
print(f"Combined:        {total_speedup:.0f}x total speedup")

---
## Step 5: The SmolVLA Attention Mask in Detail

SmolVLA's action expert alternates between two types of attention:
- **Cross-Attention (CA):** Every action token attends to ALL VLM tokens (full rectangular mask)
- **Self-Attention (SA):** Each action token only attends to previous action tokens (causal triangular mask)

This alternation lets the model ground actions in vision (CA) while maintaining temporal coherence (SA).

In [ ]:
n_action_tokens = 50  # action horizon
n_vlm_tokens = 64     # after PixelShuffle compression

# Cross-Attention mask: 50 action tokens x 64 VLM tokens (all 1s = full attention)
ca_mask = np.ones((n_action_tokens, n_vlm_tokens))

# Self-Attention mask: 50 x 50 lower triangular (causal)
sa_mask = np.tril(np.ones((n_action_tokens, n_action_tokens)))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Cross-Attention
ax = axes[0]
im0 = ax.imshow(ca_mask, cmap='YlOrBr', aspect='auto', vmin=0, vmax=1)
ax.set_title('Cross-Attention Mask (CA)\n50 action tokens x 64 VLM tokens',
             fontsize=13, color=ACCENT, fontweight='bold')
ax.set_xlabel('VLM tokens (Key/Value)', fontweight='bold')
ax.set_ylabel('Action tokens (Query)', fontweight='bold')
ax.set_xticks([0, 31, 63])
ax.set_xticklabels(['0', '31', '63'])
ax.set_yticks([0, 24, 49])
ax.set_yticklabels(['0', '24', '49'])

# Annotation
ax.text(32, 25, 'Every action token\nsees ALL VLM tokens',
        ha='center', va='center', fontsize=12, fontweight='bold',
        color='white', bbox=dict(boxstyle='round,pad=0.3', facecolor=ACCENT, alpha=0.8))

# Self-Attention
ax = axes[1]
im1 = ax.imshow(sa_mask, cmap='PuBuGn', aspect='auto', vmin=0, vmax=1)
ax.set_title('Self-Attention Mask (SA)\n50 action tokens x 50 action tokens',
             fontsize=13, color=TEAL, fontweight='bold')
ax.set_xlabel('Action tokens (Key/Value)', fontweight='bold')
ax.set_ylabel('Action tokens (Query)', fontweight='bold')
ax.set_xticks([0, 24, 49])
ax.set_xticklabels(['0', '24', '49'])
ax.set_yticks([0, 24, 49])
ax.set_yticklabels(['0', '24', '49'])

# Annotation
ax.text(35, 15, 'Causal: action t\nsees actions 1..t-1',
        ha='center', va='center', fontsize=12, fontweight='bold',
        color='white', bbox=dict(boxstyle='round,pad=0.3', facecolor=TEAL, alpha=0.8))

plt.suptitle('SmolVLA Attention Masks', fontsize=15, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

# Visualize the 6-layer alternation pattern
fig, ax = plt.subplots(figsize=(14, 3))

layer_types = ['CA', 'SA', 'CA', 'SA', 'CA', 'SA']
layer_colors = [ACCENT if t == 'CA' else TEAL for t in layer_types]

for i, (ltype, lcolor) in enumerate(zip(layer_types, layer_colors)):
    rect = plt.Rectangle((i * 2, 0), 1.6, 1, facecolor=lcolor, edgecolor='white',
                          linewidth=2, alpha=0.9)
    ax.add_patch(rect)
    ax.text(i * 2 + 0.8, 0.5, f'Layer {i+1}\n{ltype}',
            ha='center', va='center', fontsize=13, fontweight='bold', color='white')
    
    # Draw arrow between layers
    if i < 5:
        ax.annotate('', xy=(i * 2 + 2, 0.5), xytext=(i * 2 + 1.6, 0.5),
                    arrowprops=dict(arrowstyle='->', color='#555', lw=2))

ax.set_xlim(-0.2, 12)
ax.set_ylim(-0.3, 1.5)
ax.set_title('SmolVLA Action Expert: 6 Alternating Layers', fontsize=14, color=ACCENT, fontweight='bold')
ax.axis('off')

# Legend
ca_patch = mpatches.Patch(color=ACCENT, label='Cross-Attention (grounds in vision)')
sa_patch = mpatches.Patch(color=TEAL, label='Self-Attention (temporal coherence)')
ax.legend(handles=[ca_patch, sa_patch], loc='upper right', fontsize=11)

plt.tight_layout()
plt.show()

print("Pattern: CA -> SA -> CA -> SA -> CA -> SA")
print("  CA layers: action tokens attend to VLM features (what does the robot see?)")
print("  SA layers: action tokens attend to previous actions (temporal consistency)")

---
## Step 6: Building the Full SmolVLA Action Expert

Now we build the complete action expert with:
- Sinusoidal noise embedding for flow matching timestep tau
- 6 interleaved Cross-Attention + Self-Attention blocks
- Output projection to action dimensions
- Full flow matching training integration

In [ ]:
class SinusoidalEmbedding(nn.Module):
    """Sinusoidal embedding for flow matching timestep tau."""
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.proj = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
        )
    
    def forward(self, tau):
        # tau: [B] in [0, 1]
        half_dim = self.d_model // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=tau.device, dtype=tau.dtype) * -emb)
        emb = tau.unsqueeze(-1) * emb.unsqueeze(0)  # [B, half_dim]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)  # [B, d_model]
        return self.proj(emb)  # [B, d_model]


class CrossAttentionBlock(nn.Module):
    """Cross-attention: Q from actions, KV from VLM features."""
    def __init__(self, d_model=256, n_heads=4):
        super().__init__()
        self.ln_q = nn.LayerNorm(d_model)
        self.ln_kv = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln_ff = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model),
        )
    
    def forward(self, actions, vlm_features):
        # actions: [B, T_a, d_model], vlm_features: [B, T_v, d_model]
        q = self.ln_q(actions)
        kv = self.ln_kv(vlm_features)
        h, _ = self.attn(q, kv, kv)  # Q from actions, KV from VLM
        actions = actions + h
        actions = actions + self.ff(self.ln_ff(actions))
        return actions


class SelfAttentionBlock(nn.Module):
    """Causal self-attention over action tokens."""
    def __init__(self, d_model=256, n_heads=4):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model),
        )
    
    def forward(self, actions):
        # actions: [B, T_a, d_model]
        T = actions.shape[1]
        causal_mask = torch.triu(torch.ones(T, T, device=actions.device), diagonal=1).bool()
        h = self.ln1(actions)
        h, _ = self.attn(h, h, h, attn_mask=causal_mask)
        actions = actions + h
        actions = actions + self.ff(self.ln2(actions))
        return actions


class SmolVLAActionExpert(nn.Module):
    """Complete SmolVLA Action Expert.
    
    - Sinusoidal noise embedding for flow matching tau
    - 6 interleaved CA + SA blocks
    - Output projection to action_dim
    """
    def __init__(self, action_dim=7, action_horizon=50, d_model=256,
                 d_vlm=256, n_heads=4, n_blocks=3):
        super().__init__()
        self.action_dim = action_dim
        self.action_horizon = action_horizon
        self.d_model = d_model
        
        # Project noisy actions to d_model
        self.action_proj = nn.Linear(action_dim, d_model)
        
        # Noise embedding (sinusoidal for flow matching tau)
        self.tau_embed = SinusoidalEmbedding(d_model)
        
        # Project VLM features to d_model if different
        self.vlm_proj = nn.Linear(d_vlm, d_model) if d_vlm != d_model else nn.Identity()
        
        # 6 interleaved blocks: CA, SA, CA, SA, CA, SA (n_blocks pairs)
        self.ca_blocks = nn.ModuleList([CrossAttentionBlock(d_model, n_heads) for _ in range(n_blocks)])
        self.sa_blocks = nn.ModuleList([SelfAttentionBlock(d_model, n_heads) for _ in range(n_blocks)])
        
        # Output projection
        self.out_ln = nn.LayerNorm(d_model)
        self.out_proj = nn.Linear(d_model, action_dim)
    
    def forward(self, noisy_actions, tau, vlm_features):
        """
        Args:
            noisy_actions: [B, 50, action_dim] — noisy action trajectory
            tau: [B] — flow matching timestep in [0, 1]
            vlm_features: [B, 64, d_vlm] — VLM backbone output
        Returns:
            velocity: [B, 50, action_dim] — predicted flow velocity
        """
        B = noisy_actions.shape[0]
        
        # Project noisy actions to d_model
        h = self.action_proj(noisy_actions)  # [B, 50, d_model]
        
        # Add tau embedding (broadcast to all action tokens)
        tau_emb = self.tau_embed(tau)  # [B, d_model]
        h = h + tau_emb.unsqueeze(1)  # [B, 50, d_model]
        
        # Project VLM features
        vlm = self.vlm_proj(vlm_features)  # [B, 64, d_model]
        
        # 6 alternating layers: CA, SA, CA, SA, CA, SA
        for ca, sa in zip(self.ca_blocks, self.sa_blocks):
            h = ca(h, vlm)   # Cross-attend to vision
            h = sa(h)        # Causal self-attend over actions
        
        # Output projection
        velocity = self.out_proj(self.out_ln(h))  # [B, 50, action_dim]
        return velocity


# Build the action expert
action_dim = 7  # typical robot action dim (e.g., 6 DoF + gripper)
action_horizon = 50
d_model_ae = 256
d_vlm = 256

action_expert = SmolVLAActionExpert(
    action_dim=action_dim,
    action_horizon=action_horizon,
    d_model=d_model_ae,
    d_vlm=d_vlm,
    n_heads=4,
    n_blocks=3,
).to(device)

print(f"SmolVLA Action Expert built!")
print(f"  action_dim={action_dim}, horizon={action_horizon}, d_model={d_model_ae}")
print(f"  Layers: 6 (3x CA + 3x SA, alternating)")

# Parameter count breakdown
print(f"\nParameter breakdown:")
total = 0
for name, child in action_expert.named_children():
    n = sum(p.numel() for p in child.parameters())
    total += n
    print(f"  {name:>15s}: {n:>10,} params")
print(f"  {'TOTAL':>15s}: {total:>10,} params")

In [ ]:
# Integrate with flow matching training: one training step with full shape trace

# Simulate inputs
B = 4
gt_actions = torch.randn(B, action_horizon, action_dim, device=device)  # ground truth actions
vlm_features = torch.randn(B, 64, d_vlm, device=device)                # VLM output

print("=== Flow Matching Training Step ===")
print(f"\n1. Ground truth actions:  {list(gt_actions.shape)}")
print(f"   VLM features:          {list(vlm_features.shape)}")

# Sample tau ~ Uniform(0, 1)
tau = torch.rand(B, device=device)
print(f"\n2. Sample tau ~ U(0,1):   {list(tau.shape)}, values: {tau.cpu().numpy().round(3)}")

# Sample noise
noise = torch.randn_like(gt_actions)
print(f"   Sample noise:          {list(noise.shape)}")

# Interpolate: x_tau = (1 - tau) * noise + tau * gt_actions
tau_expand = tau.view(B, 1, 1)  # [B, 1, 1] for broadcasting
x_tau = (1 - tau_expand) * noise + tau_expand * gt_actions
print(f"\n3. Interpolate x_tau:     {list(x_tau.shape)}")
print(f"   x_tau = (1-tau)*noise + tau*gt_actions")

# Target velocity: v = gt_actions - noise
target_velocity = gt_actions - noise
print(f"\n4. Target velocity:       {list(target_velocity.shape)}")
print(f"   v_target = gt_actions - noise")

# Forward pass through action expert
pred_velocity = action_expert(x_tau, tau, vlm_features)
print(f"\n5. Predicted velocity:    {list(pred_velocity.shape)}")

# Compute loss
loss = F.mse_loss(pred_velocity, target_velocity)
print(f"\n6. MSE loss:              {loss.item():.4f}")

# Backward pass
loss.backward()
print(f"   Backward pass complete.")

print(f"\n=== Shape Summary ===")
print(f"   gt_actions:     [B={B}, T={action_horizon}, A={action_dim}]")
print(f"   x_tau (noisy):  [B={B}, T={action_horizon}, A={action_dim}]")
print(f"   tau:            [B={B}]")
print(f"   vlm_features:   [B={B}, N=64, D={d_vlm}]")
print(f"   pred_velocity:  [B={B}, T={action_horizon}, A={action_dim}]")

---
## Step 7: Asynchronous Inference Simulation

In synchronous inference, the robot sits idle while the model thinks. SmolVLA uses **action chunking** to overlap computation with execution:
- While the robot executes chunk N (a buffer of predicted actions), the model encodes the next observation and predicts chunk N+1.
- Result: ~30% wall-clock speedup.

In [ ]:
# Timing constants (in seconds) — realistic for a small VLA on edge device
T_OBSERVE  = 0.005   # 5 ms — capture camera image
T_ENCODE   = 0.200   # 200 ms — SigLIP + PixelShuffle + VLM layers
T_FLOW     = 0.100   # 100 ms — flow matching denoising (10 Euler steps)
T_EXECUTE  = 0.500   # 500 ms — execute action chunk on robot


class SyncInferenceSimulator:
    """Synchronous: observe -> encode -> flow_match -> execute -> repeat."""
    def __init__(self):
        self.timeline = []
    
    def run(self, n_cycles=10):
        t = 0.0
        for i in range(n_cycles):
            # Observe
            self.timeline.append(('observe', i, t, t + T_OBSERVE))
            t += T_OBSERVE
            # Encode
            self.timeline.append(('encode', i, t, t + T_ENCODE))
            t += T_ENCODE
            # Flow match
            self.timeline.append(('flow_match', i, t, t + T_FLOW))
            t += T_FLOW
            # Execute
            self.timeline.append(('execute', i, t, t + T_EXECUTE))
            t += T_EXECUTE
        return t


class AsyncInferenceSimulator:
    """Asynchronous: execute chunk N while encoding chunk N+1 in parallel."""
    def __init__(self):
        self.timeline = []
    
    def run(self, n_cycles=10):
        t = 0.0
        
        # First cycle is synchronous (need initial actions)
        self.timeline.append(('observe', 0, t, t + T_OBSERVE))
        t += T_OBSERVE
        self.timeline.append(('encode', 0, t, t + T_ENCODE))
        t += T_ENCODE
        self.timeline.append(('flow_match', 0, t, t + T_FLOW))
        t += T_FLOW
        
        for i in range(1, n_cycles):
            exec_start = t
            exec_end = t + T_EXECUTE
            self.timeline.append(('execute', i - 1, exec_start, exec_end))
            
            # In parallel: observe + encode + flow_match for next chunk
            obs_start = exec_start
            obs_end = obs_start + T_OBSERVE
            self.timeline.append(('observe', i, obs_start, obs_end))
            
            enc_start = obs_end
            enc_end = enc_start + T_ENCODE
            self.timeline.append(('encode', i, enc_start, enc_end))
            
            flow_start = enc_end
            flow_end = flow_start + T_FLOW
            self.timeline.append(('flow_match', i, flow_start, flow_end))
            
            # Wait for execution to finish (it's the longer one)
            t = max(exec_end, flow_end)
        
        # Final execution
        self.timeline.append(('execute', n_cycles - 1, t, t + T_EXECUTE))
        t += T_EXECUTE
        return t


# Run both simulators
n_cycles = 10
sync_sim = SyncInferenceSimulator()
sync_total = sync_sim.run(n_cycles)

async_sim = AsyncInferenceSimulator()
async_total = async_sim.run(n_cycles)

print(f"Sync total:  {sync_total:.2f}s for {n_cycles} cycles")
print(f"Async total: {async_total:.2f}s for {n_cycles} cycles")
print(f"Speedup:     {(1 - async_total / sync_total) * 100:.0f}% faster")

In [ ]:
# Gantt chart visualization
phase_colors = {
    'observe':    WARM,
    'encode':     BLUE,
    'flow_match': PURPLE,
    'execute':    TEAL,
}

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

for ax, sim, title in [(axes[0], sync_sim, f'Synchronous Inference ({sync_total:.2f}s total)'),
                        (axes[1], async_sim, f'Asynchronous Inference ({async_total:.2f}s total)')]:
    y_positions = {}  # phase -> y position
    phases_ordered = ['observe', 'encode', 'flow_match', 'execute']
    for idx, phase in enumerate(phases_ordered):
        y_positions[phase] = idx
    
    for phase, cycle, start, end in sim.timeline:
        y = y_positions[phase]
        ax.barh(y, end - start, left=start, height=0.6,
                color=phase_colors[phase], edgecolor='white', linewidth=1,
                alpha=0.85)
        # Label with cycle number (only if wide enough)
        if (end - start) > 0.03:
            ax.text((start + end) / 2, y, f'{cycle}', ha='center', va='center',
                    fontsize=8, fontweight='bold', color='white')
    
    ax.set_yticks(range(len(phases_ordered)))
    ax.set_yticklabels([p.replace('_', '\n') for p in phases_ordered], fontweight='bold')
    ax.set_title(title, fontsize=13, color=ACCENT, fontweight='bold')
    ax.grid(True, alpha=0.2, axis='x')
    ax.set_xlim(-0.1, max(sync_total, async_total) + 0.3)

axes[1].set_xlabel('Time (seconds)', fontweight='bold')

# Legend
legend_patches = [mpatches.Patch(color=c, label=p.replace('_', ' ').title())
                  for p, c in phase_colors.items()]
axes[0].legend(handles=legend_patches, loc='upper right', fontsize=10)

plt.suptitle('Sync vs Async Inference Timeline', fontsize=15, color=ACCENT, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Sync:  {sync_total:.2f}s — robot sits idle during encode+flow_match ({T_ENCODE + T_FLOW:.3f}s per cycle)")
print(f"Async: {async_total:.2f}s — encode+flow_match overlaps with execute")
print(f"")
print(f"WHY action chunking is required:")
print(f"  The robot needs a BUFFER of future actions to execute while the model thinks.")
print(f"  Without chunking, the robot has only 1 action and must wait for the next prediction.")
print(f"  With chunk_size=50, the robot has 50 actions (~500ms) of buffer = enough time to")
print(f"  observe, encode, and run flow matching for the next chunk.")

---
## Step 8: End-to-End SmolVLA Forward Pass

Let's wire everything together into a complete SmolVLA pipeline:
1. Image -> SigLIP (simulated) -> 256 patches x 1152-d
2. PixelShuffle -> 64 tokens x 4608-d -> project to d_model
3. Bottom N/2 VLM layers (6 transformer blocks)
4. Action expert: noisy actions + VLM features -> flow matching -> predicted velocity

In [ ]:
class SmolVLAPipeline(nn.Module):
    """Complete SmolVLA forward pass pipeline."""
    def __init__(self, siglip_dim=1152, d_model=256, n_vlm_layers=6,
                 action_dim=7, action_horizon=50):
        super().__init__()
        self.siglip_dim = siglip_dim
        self.d_model = d_model
        
        # Stage 1: PixelShuffle compression (256 -> 64 tokens)
        self.pixel_shuffle = PixelShuffleCompressor(d_model=siglip_dim, grid_size=16, downsample_factor=2)
        
        # Project compressed tokens (1152*4 = 4608) down to d_model
        self.vision_proj = nn.Linear(siglip_dim * 4, d_model)
        
        # Stage 2: Bottom N/2 VLM layers
        self.vlm_layers = nn.ModuleList([
            TransformerBlock(d_model=d_model, n_heads=4, d_ff=d_model * 4)
            for _ in range(n_vlm_layers)
        ])
        
        # Stage 3: Action expert
        self.action_expert = SmolVLAActionExpert(
            action_dim=action_dim,
            action_horizon=action_horizon,
            d_model=d_model,
            d_vlm=d_model,
            n_heads=4,
            n_blocks=3,
        )
    
    def forward(self, siglip_patches, noisy_actions, tau, verbose=False):
        """
        Args:
            siglip_patches: [B, 256, 1152] — SigLIP output
            noisy_actions:  [B, 50, action_dim]
            tau:            [B]
        """
        if verbose:
            print("=== SmolVLA Forward Pass ===")
            print(f"\n  Input SigLIP patches:   {list(siglip_patches.shape)}")
        
        # Stage 1: PixelShuffle compression
        compressed = self.pixel_shuffle(siglip_patches)  # [B, 64, 4608]
        if verbose:
            print(f"  After PixelShuffle:     {list(compressed.shape)}")
        
        # Project to d_model
        vlm_input = self.vision_proj(compressed)  # [B, 64, d_model]
        if verbose:
            print(f"  After projection:       {list(vlm_input.shape)}")
        
        # Stage 2: Bottom N/2 VLM layers
        vlm_out = vlm_input
        for i, layer in enumerate(self.vlm_layers):
            vlm_out = layer(vlm_out)
        if verbose:
            print(f"  After {len(self.vlm_layers)} VLM layers:      {list(vlm_out.shape)}")
            print(f"  Noisy actions input:    {list(noisy_actions.shape)}")
            print(f"  Tau:                    {list(tau.shape)}")
        
        # Stage 3: Action expert
        velocity = self.action_expert(noisy_actions, tau, vlm_out)
        if verbose:
            print(f"  Predicted velocity:     {list(velocity.shape)}")
        
        return velocity


# Build the full pipeline
pipeline = SmolVLAPipeline(
    siglip_dim=1152,
    d_model=256,
    n_vlm_layers=6,
    action_dim=7,
    action_horizon=50,
).to(device)

total_params = sum(p.numel() for p in pipeline.parameters())
print(f"SmolVLA Pipeline: {total_params:,} parameters\n")

# Simulate a full forward pass
B = 2

# Step 1: Simulate SigLIP output (256 patches x 1152-d)
siglip_output = torch.randn(B, 256, 1152, device=device)

# Step 2: Noisy actions for flow matching
noisy_actions = torch.randn(B, 50, 7, device=device)
tau = torch.rand(B, device=device)

# Full forward pass with shape trace
pipeline.eval()
with torch.no_grad():
    velocity = pipeline(siglip_output, noisy_actions, tau, verbose=True)

# Benchmark forward pass time
print(f"\n=== Benchmark (CPU timing) ===")
pipeline.eval()
n_runs = 10

# Warm up
with torch.no_grad():
    for _ in range(3):
        _ = pipeline(siglip_output, noisy_actions, tau)

times = []
for _ in range(n_runs):
    if device == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = pipeline(siglip_output, noisy_actions, tau)
    if device == 'cuda':
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    times.append(t1 - t0)

mean_time = np.mean(times) * 1000
std_time = np.std(times) * 1000
print(f"  SmolVLA forward pass: {mean_time:.1f} +/- {std_time:.1f} ms (B={B}, {n_runs} runs)")

# Compare with hypothetical OpenVLA autoregressive timing
# OpenVLA: 256 tokens decoded one at a time, each requiring a full forward pass
openvla_est = mean_time * 256 / 50  # rough scaling: 256 autoregressive steps vs 50 parallel
print(f"  Hypothetical OpenVLA (autoregressive, 256 steps): ~{openvla_est:.0f} ms")
print(f"  SmolVLA speedup: ~{openvla_est / mean_time:.0f}x faster")

---
## Step 9: Summary

### Efficiency Gains Table

| Trick | Savings | Mechanism |
|-------|---------|----------|
| **PixelShuffle** | 16x attention | 256 -> 64 tokens |
| **Layer Skip** | 2x forward pass | 12 -> 6 layers |
| **Async Inference** | ~30% wall-clock | parallel encode + execute |

### SmolVLA vs OpenVLA

| Property | OpenVLA | SmolVLA |
|----------|---------|--------|
| Parameters | 7B | 500M |
| Vision tokens | 256 | 64 (PixelShuffle) |
| VLM layers used | All | Bottom half |
| Action decoding | Autoregressive (slow) | Flow matching (parallel) |
| Action expert | None (LLM head) | Dedicated 6-layer CA/SA |
| Inference | Synchronous | Asynchronous (action chunking) |
| Edge deployment | No (needs A100) | Yes (runs on Jetson) |

The key insight: **SmolVLA is not a smaller version of OpenVLA.** It is a fundamentally different architecture that trades autoregressive generality for efficient, parallel action generation.

---

## Exercises

**Exercise 1: PixelShuffle Compression Factors**

Modify `PixelShuffleCompressor` to support compression factors of 2x, 3x, and 4x (i.e., `downsample_factor` = 2, 4, 8 resulting in 64, 16, and 4 tokens respectively). For each, pass the tokens through a small 4-layer transformer and measure the output quality using cosine similarity with the uncompressed 256-token version. Plot: token count vs reconstruction quality.

**Exercise 2: Which Layers to Skip**

Using the 12-layer `ToyVLMBackbone`, compare three skipping strategies:
- Skip top N/2 (keep layers 1-6) — SmolVLA's approach
- Skip even layers (keep 1, 3, 5, 7, 9, 11)
- Skip bottom N/2 (keep layers 7-12)

For each, measure CKA similarity between the skipped model's output and the full 12-layer output. Which strategy preserves the most information?

**Exercise 3: Action Chunk Size Effect on Async Speedup**

Modify `AsyncInferenceSimulator` so that `T_EXECUTE` scales with chunk size: `T_EXECUTE = chunk_size * 0.01s`. Test chunk sizes of 10, 25, 50, and 100. Plot the async speedup percentage vs chunk size. At what chunk size does async inference stop helping? (Hint: when `T_EXECUTE < T_ENCODE + T_FLOW`, the computation is the bottleneck, not execution.)

**Exercise 4: Euler Steps Sweet Spot**

Implement Euler integration for flow matching inference in `SmolVLAActionExpert`:
```python
x = noise
for i in range(n_euler_steps):
    tau = i / n_euler_steps
    v = action_expert(x, tau, vlm_features)
    x = x + v * (1 / n_euler_steps)
```
Test with 5, 10, 20, and 50 Euler steps. Measure both (a) the MSE between predicted and ground-truth actions and (b) wall-clock inference time. Plot quality vs time — where is the sweet spot?

**Exercise 5 (Challenge): Adaptive Async Inference**

Build an `AdaptiveAsyncSimulator` that monitors the predicted velocity magnitude at each step. When `||v|| > threshold`, the robot is making large corrections (high uncertainty) — re-compute immediately. When `||v|| < threshold`, the robot is coasting — keep executing the current chunk. Simulate 50 cycles with a velocity profile that alternates between high and low uncertainty phases. Compare the total time and number of re-computations vs the fixed-schedule async baseline.